Move into repo

In [2]:
!git clone https://github.com/ChloeMVan/inf-deep-learning.git
%cd inf-deep-learning

Cloning into 'inf-deep-learning'...
remote: Enumerating objects: 1786, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 1786 (delta 15), reused 20 (delta 9), pack-reused 1755 (from 1)
Receiving objects: 100% (1786/1786), 1.47 GiB | 50.46 MiB/s, done.
Resolving deltas: 100% (33/33), done.
Updating files: 100% (1093/1093), done.
/content/inf-deep-learning


In [3]:
!git checkout baseline
!git branch

Branch 'baseline' set up to track remote branch 'baseline' from 'origin'.
Switched to a new branch 'baseline'
* baseline
  main


In [4]:
import os
if not os.path.exists('FaceForensics/dataset/extract_compressed_videos.py'):
    !rm -rf FaceForensics
    !git clone https://github.com/ondyari/FaceForensics.git FaceForensics

Cloning into 'FaceForensics'...
remote: Enumerating objects: 414, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 414 (delta 29), reused 14 (delta 14), pack-reused 368 (from 1)
Receiving objects: 100% (414/414), 59.81 MiB | 58.05 MiB/s, done.
Resolving deltas: 100% (166/166), done.


In [5]:
!git status

On branch baseline
Your branch is up to date with 'origin/baseline'.

nothing to commit, working tree clean


In [6]:
%%writefile FaceForensics/download.py
#!/usr/bin/env python
""" Downloads FaceForensics++ and Deep Fake Detection public data release
Example usage:
    see -h or https://github.com/ondyari/FaceForensics
"""
# -*- coding: utf-8 -*-
import argparse
import os
import urllib
import urllib.request
import tempfile
import time
import sys
import json
import random
from tqdm import tqdm
from os.path import join


# URLs and filenames
FILELIST_URL = 'misc/filelist.json'
DEEPFEAKES_DETECTION_URL = 'misc/deepfake_detection_filenames.json'
DEEPFAKES_MODEL_NAMES = ['decoder_A.h5', 'decoder_B.h5', 'encoder.h5',]

# Parameters
DATASETS = {
    'original_youtube_videos': 'misc/downloaded_youtube_videos.zip',
    'original_youtube_videos_info': 'misc/downloaded_youtube_videos_info.zip',
    'original': 'original_sequences/youtube',
    'DeepFakeDetection_original': 'original_sequences/actors',
    'Deepfakes': 'manipulated_sequences/Deepfakes',
    'DeepFakeDetection': 'manipulated_sequences/DeepFakeDetection',
    'Face2Face': 'manipulated_sequences/Face2Face',
    'FaceShifter': 'manipulated_sequences/FaceShifter',
    'FaceSwap': 'manipulated_sequences/FaceSwap',
    'NeuralTextures': 'manipulated_sequences/NeuralTextures'
    }
ALL_DATASETS = ['original', 'DeepFakeDetection_original', 'Deepfakes',
                'DeepFakeDetection', 'Face2Face', 'FaceShifter', 'FaceSwap',
                'NeuralTextures']
COMPRESSION = ['raw', 'c23', 'c40']
TYPE = ['videos', 'masks', 'models']
SERVERS = ['EU', 'EU2', 'CA']


def parse_args():
    parser = argparse.ArgumentParser(
        description='Downloads FaceForensics v2 public data release.',
        formatter_class=argparse.ArgumentDefaultsHelpFormatter
    )
    parser.add_argument('output_path', type=str, help='Output directory.')
    parser.add_argument('-d', '--dataset', type=str, default='all',
                        help='Which dataset to download, either pristine or '
                             'manipulated data or the downloaded youtube '
                             'videos.',
                        choices=list(DATASETS.keys()) + ['all']
                        )
    parser.add_argument('-c', '--compression', type=str, default='raw',
                        help='Which compression degree. All videos '
                             'have been generated with h264 with a varying '
                             'codec. Raw (c0) videos are lossless compressed.',
                        choices=COMPRESSION
                        )
    parser.add_argument('-t', '--type', type=str, default='videos',
                        help='Which file type, i.e. videos, masks, for our '
                             'manipulation methods, models, for Deepfakes.',
                        choices=TYPE
                        )
    parser.add_argument('-n', '--num_videos', type=int, default=None,
                        help='Select a number of videos number to '
                             "download if you don't want to download the full"
                             ' dataset.')
    parser.add_argument('--server', type=str, default='EU',
                        help='Server to download the data from. If you '
                             'encounter a slow download speed, consider '
                             'changing the server.',
                        choices=SERVERS
                        )
    args = parser.parse_args()

    # URLs
    server = args.server
    if server == 'EU':
        server_url = 'http://canis.vc.in.tum.de:8100/'
    elif server == 'EU2':
        server_url = 'http://kaldir.vc.in.tum.de/faceforensics/'
    elif server == 'CA':
        server_url = 'http://falas.cmpt.sfu.ca:8100/'
    else:
        raise Exception('Wrong server name. Choices: {}'.format(str(SERVERS)))
    args.tos_url = server_url + 'webpage/FaceForensics_TOS.pdf'
    args.base_url = server_url + 'v3/'
    args.deepfakes_model_url = server_url + 'v3/manipulated_sequences/' + \
                               'Deepfakes/models/'

    return args


def download_files(filenames, base_url, output_path, report_progress=True):
    os.makedirs(output_path, exist_ok=True)
    if report_progress:
        filenames = tqdm(filenames)
    for filename in filenames:
        download_file(base_url + filename, join(output_path, filename))


def reporthook(count, block_size, total_size):
    global start_time
    if count == 0:
        start_time = time.time()
        return
    duration = time.time() - start_time
    progress_size = int(count * block_size)
    speed = int(progress_size / (1024 * duration))
    percent = int(count * block_size * 100 / total_size)
    sys.stdout.write("\rProgress: %d%%, %d MB, %d KB/s, %d seconds passed" %
                     (percent, progress_size / (1024 * 1024), speed, duration))
    sys.stdout.flush()


def download_file(url, out_file, report_progress=False):
    out_dir = os.path.dirname(out_file)
    if not os.path.isfile(out_file):
        fh, out_file_tmp = tempfile.mkstemp(dir=out_dir)
        f = os.fdopen(fh, 'w')
        f.close()
        if report_progress:
            urllib.request.urlretrieve(url, out_file_tmp,
                                       reporthook=reporthook)
        else:
            urllib.request.urlretrieve(url, out_file_tmp)
        os.rename(out_file_tmp, out_file)
    else:
        tqdm.write('WARNING: skipping download of existing file ' + out_file)


def main(args):
    # TOS
    print('By pressing any key to continue you confirm that you have agreed '\
          'to the FaceForensics terms of use as described at:')
    print(args.tos_url)
    print('***')
    print('Press any key to continue, or CTRL-C to exit.')
    _ = input('')

    # Extract arguments
    c_datasets = [args.dataset] if args.dataset != 'all' else ALL_DATASETS
    c_type = args.type
    c_compression = args.compression
    num_videos = args.num_videos
    output_path = args.output_path
    os.makedirs(output_path, exist_ok=True)

    # Check for special dataset cases
    for dataset in c_datasets:
        dataset_path = DATASETS[dataset]
        if 'original_youtube_videos' in dataset:
            print('Downloading original youtube videos.')
            if not 'info' in dataset_path:
                print('Please be patient, this may take a while (~40gb)')
                suffix = ''
            else:
                suffix = 'info'
            download_file(args.base_url + '/' + dataset_path,
                          out_file=join(output_path,
                                        'downloaded_videos{}.zip'.format(
                                            suffix)),
                          report_progress=True)
            return

        print('Downloading {} of dataset "{}"'.format(c_type, dataset_path))

        if 'DeepFakeDetection' in dataset_path or 'actors' in dataset_path:
            filepaths = json.loads(urllib.request.urlopen(args.base_url + '/' +
                DEEPFEAKES_DETECTION_URL).read().decode("utf-8"))
            if 'actors' in dataset_path:
                filelist = filepaths['actors']
            else:
                filelist = filepaths['DeepFakesDetection']
        elif 'original' in dataset_path:
            file_pairs = json.loads(urllib.request.urlopen(args.base_url + '/' +
                FILELIST_URL).read().decode("utf-8"))
            filelist = []
            for pair in file_pairs:
                filelist += pair
        else:
            file_pairs = json.loads(urllib.request.urlopen(args.base_url + '/' +
                FILELIST_URL).read().decode("utf-8"))
            filelist = []
            for pair in file_pairs:
                filelist.append('_'.join(pair))
                if c_type != 'models':
                    filelist.append('_'.join(pair[::-1]))

        if num_videos is not None and num_videos > 0:
            print('Downloading the first {} videos'.format(num_videos))
            filelist = filelist[:num_videos]

        dataset_videos_url = args.base_url + '{}/{}/{}/'.format(
            dataset_path, c_compression, c_type)
        dataset_mask_url = args.base_url + '{}/{}/videos/'.format(
            dataset_path, 'masks', c_type)

        if c_type == 'videos':
            dataset_output_path = join(output_path, dataset_path, c_compression,
                                       c_type)
            print('Output path: {}'.format(dataset_output_path))
            filelist = [filename + '.mp4' for filename in filelist]
            download_files(filelist, dataset_videos_url, dataset_output_path)
        elif c_type == 'masks':
            dataset_output_path = join(output_path, dataset_path, c_type,
                                       'videos')
            print('Output path: {}'.format(dataset_output_path))
            if 'original' in dataset:
                if args.dataset != 'all':
                    print('Only videos available for original data. Aborting.')
                    return
                else:
                    print('Only videos available for original data. Skipping original.\n')
                    continue
            if 'FaceShifter' in dataset:
                print('Masks not available for FaceShifter. Aborting.')
                return
            filelist = [filename + '.mp4' for filename in filelist]
            download_files(filelist, dataset_mask_url, dataset_output_path)
        else:
            if dataset != 'Deepfakes' and c_type == 'models':
                print('Models only available for Deepfakes. Aborting')
                return
            dataset_output_path = join(output_path, dataset_path, c_type)
            print('Output path: {}'.format(dataset_output_path))
            for folder in tqdm(filelist):
                folder_filelist = DEEPFAKES_MODEL_NAMES
                folder_base_url = args.deepfakes_model_url + folder + '/'
                folder_dataset_output_path = join(dataset_output_path, folder)
                download_files(folder_filelist, folder_base_url,
                               folder_dataset_output_path,
                               report_progress=False)


if __name__ == "__main__":
    args = parse_args()
    main(args)

Writing FaceForensics/download.py


In [7]:
!ls FaceForensics/download.py
!ls FaceForensics/dataset/extract_compressed_videos.py

FaceForensics/download.py
FaceForensics/dataset/extract_compressed_videos.py


In [8]:
!sed -i 's|OUTPUT_BASE=~/Documents/INF-Deep_Learning/FF_data|OUTPUT_BASE=/content/FF_data|g' download_ff_data.sh

# Verify
!grep "OUTPUT_BASE" download_ff_data.sh

OUTPUT_BASE=/content/FF_data
echo "  Output:      $OUTPUT_BASE"
run_download "Original (YouTube real videos)"   "original"                   "$OUTPUT_BASE/real"
run_download "Face2Face (manipulated)"          "Face2Face"                  "$OUTPUT_BASE/Face2Face"
run_download "FaceSwap (manipulated)"           "FaceSwap"                   "$OUTPUT_BASE/FaceSwap"
run_download "NeuralTextures (manipulated)"     "NeuralTextures"             "$OUTPUT_BASE/NeuralTextures"
run_download "Deepfakes (manipulated)"          "Deepfakes"                  "$OUTPUT_BASE/fake"
run_download "DFD Real (Google actor videos)"   "DeepFakeDetection_original" "$OUTPUT_BASE/DFD_real"
run_download "DFD Fake (Google deepfakes)"      "DeepFakeDetection"          "$OUTPUT_BASE/DFD_fake"
run_extract "Original (YouTube real videos)"   "original"                   "$OUTPUT_BASE/real"
run_extract "Face2Face (manipulated)"          "Face2Face"                  "$OUTPUT_BASE/Face2Face"
run_extract "FaceSwap (manipulate

In [9]:
!bash download_ff_data.sh --sample

  FaceForensics++ Downloader + Extractor
  Mode:        SAMPLE (1 video each)
  Compression: c40
  Server:      EU2
  Output:      /content/FF_data

  STEP 1: Downloading videos...

----------------------------------------------
Downloading: Original (YouTube real videos)
  Dataset: original
  Output:  /content/FF_data/real
----------------------------------------------
By pressing any key to continue you confirm that you have agreed to the FaceForensics terms of use as described at:
http://kaldir.vc.in.tum.de/faceforensics/webpage/FaceForensics_TOS.pdf
***
Press any key to continue, or CTRL-C to exit.
Output path: /content/FF_data/real/original_sequences/youtube/c40/videos
100% 1/1 [00:00<00:00,  1.03it/s]
✓ Done: Original (YouTube real videos)

----------------------------------------------
Downloading: Face2Face (manipulated)
  Dataset: Face2Face
  Output:  /content/FF_data/Face2Face
----------------------------------------------
By pressing any key to continue you confirm that you 

In [10]:
all_dirs = [
    '/content/FF_data/real/original_sequences/youtube/c40/images',
    '/content/FF_data/fake/manipulated_sequences/Deepfakes/c40/images',
    '/content/FF_data/DFD_real/original_sequences/actors/c40/images',
    '/content/FF_data/DFD_fake/manipulated_sequences/DeepFakeDetection/c40/images',
    '/content/FF_data/Face2Face/manipulated_sequences/Face2Face/c40/images',
    '/content/FF_data/FaceSwap/manipulated_sequences/FaceSwap/c40/images',
    '/content/FF_data/NeuralTextures/manipulated_sequences/NeuralTextures/c40/images'
]

for d in all_dirs:
    exists = os.path.exists(d)
    print(f"{'✓' if exists else '✗'} {d}")

✗ /content/FF_data/real/original_sequences/youtube/c40/images
✓ /content/FF_data/fake/manipulated_sequences/Deepfakes/c40/images
✗ /content/FF_data/DFD_real/original_sequences/actors/c40/images
✗ /content/FF_data/DFD_fake/manipulated_sequences/DeepFakeDetection/c40/images
✓ /content/FF_data/Face2Face/manipulated_sequences/Face2Face/c40/images
✓ /content/FF_data/FaceSwap/manipulated_sequences/FaceSwap/c40/images
✗ /content/FF_data/NeuralTextures/manipulated_sequences/NeuralTextures/c40/images


In [11]:
import os

extractions = {
    '/content/FF_data/real/original_sequences/youtube/c40/videos':
        '/content/FF_data/real/original_sequences/youtube/c40/images',
    '/content/FF_data/DFD_real/original_sequences/actors/c40/videos':
        '/content/FF_data/DFD_real/original_sequences/actors/c40/images',
    '/content/FF_data/DFD_fake/manipulated_sequences/DeepFakeDetection/c40/videos':
        '/content/FF_data/DFD_fake/manipulated_sequences/DeepFakeDetection/c40/images',
    '/content/FF_data/NeuralTextures/manipulated_sequences/NeuralTextures/c40/videos':
        '/content/FF_data/NeuralTextures/manipulated_sequences/NeuralTextures/c40/images',
}

for video_dir, output_dir in extractions.items():
    os.makedirs(output_dir, exist_ok=True)
    for video_file in os.listdir(video_dir):
        if video_file.endswith('.mp4'):
            video_name = video_file.replace('.mp4', '')
            frame_output = os.path.join(output_dir, video_name)
            os.makedirs(frame_output, exist_ok=True)
            os.system(f"ffmpeg -i '{video_dir}/{video_file}' '{frame_output}/%04d.png' -hide_banner -loglevel error")
            print(f"✓ Extracted: {video_name}")

print("All done!")

✓ Extracted: 585
✓ Extracted: 07__talking_against_wall
✓ Extracted: 01_11__talking_against_wall__9229VVZ3
✓ Extracted: 585_599
All done!


In [12]:
for d in all_dirs:
    exists = os.path.exists(d)
    print(f"{'✓' if exists else '✗'} {d}")

✓ /content/FF_data/real/original_sequences/youtube/c40/images
✓ /content/FF_data/fake/manipulated_sequences/Deepfakes/c40/images
✓ /content/FF_data/DFD_real/original_sequences/actors/c40/images
✓ /content/FF_data/DFD_fake/manipulated_sequences/DeepFakeDetection/c40/images
✓ /content/FF_data/Face2Face/manipulated_sequences/Face2Face/c40/images
✓ /content/FF_data/FaceSwap/manipulated_sequences/FaceSwap/c40/images
✓ /content/FF_data/NeuralTextures/manipulated_sequences/NeuralTextures/c40/images


# Dataset processing

In [13]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
import os
import pandas as pd
from collections import defaultdict

Image.MAX_IMAGE_PIXELS = None

In [14]:
def build_samples(all_dirs):
    samples = []

    if isinstance(all_dirs, dict):
        dirs_with_labels = all_dirs.items()
    else:
        dirs_with_labels = [
            (d, 0 if 'original' in d else 1)
            for d in all_dirs
        ]

    for directory, label in dirs_with_labels:
        for video_name in os.listdir(directory):
            video_path = os.path.join(directory, video_name)
            if not os.path.isdir(video_path):
                continue

            # Use full path as unique ID — guarantees no collisions
            unique_video_id = video_path

            for frame in sorted(os.listdir(video_path)):
                if frame.endswith('.png'):
                    frame_path = os.path.join(video_path, frame)
                    samples.append((frame_path, label, unique_video_id))

    print(f"Total frames: {len(samples)}")
    return samples

In [15]:
# split it by video
import pandas as pd
from sklearn.model_selection import train_test_split

def split_by_video(samples, test_size=0.2, random_state=42):
    """
    Splits samples into train and test by video name
    to prevent data leakage.

    Args:
        samples:      list of (frame_path, label, video_name)
        test_size:    proportion for test set (default 0.2)
        random_state: for reproducibility (default 42)

    Returns:
        train_samples, test_samples
    """
    all_videos = list(set(s[2] for s in samples))

    train_videos, test_videos = train_test_split(
        all_videos,
        test_size=test_size,
        random_state=random_state
    )

    train_samples = [s for s in samples if s[2] in train_videos]
    test_samples  = [s for s in samples if s[2] in test_videos]

    print(f"Total videos:  {len(all_videos)}")
    print(f"Train videos:  {len(train_videos)} | Train frames: {len(train_samples)}")
    print(f"Test videos:   {len(test_videos)}  | Test frames:  {len(test_samples)}")

    return train_samples, test_samples

In [16]:
# Build samples
all_samples = build_samples(all_dirs)

# Split by video
train_samples, test_samples = split_by_video(all_samples, test_size=0.2)

Total frames: 3507
Total videos:  7
Train videos:  5 | Train frames: 2789
Test videos:   2  | Test frames:  718


In [17]:
# dataset class
class BaselineDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, video_name = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label, video_name

# Transform for ResNet-50
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = BaselineDataset(train_samples, transform=transform)
test_dataset  = BaselineDataset(test_samples,  transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

# Quick check
for images, labels, video_names in train_loader:
    print(f"Batch shape: {images.shape}")  # should be (32, 3, 224, 224)
    print(f"Labels: {labels}")
    break

Batch shape: torch.Size([32, 3, 224, 224])
Labels: tensor([0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1,
        0, 0, 0, 0, 0, 0, 1, 0])


In [18]:
train_dataset = BaselineDataset(train_samples, transform=transform)
test_dataset  = BaselineDataset(test_samples,  transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

# Quick check
for images, labels, video_names in train_loader:
    print(f"Batch shape: {images.shape}")  # should be (32, 3, 224, 224)
    print(f"Labels: {labels}")
    break

Batch shape: torch.Size([32, 3, 224, 224])
Labels: tensor([0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1,
        0, 1, 1, 1, 1, 1, 1, 1])


# Baseline

Imports

In [19]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from collections import defaultdict
from PIL import Image
import pandas as pd
import numpy as np
import os

In [20]:
# load model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

# Load pretrained ResNet-50
model = models.resnet50(pretrained=True)

# Replace final layer for binary classification (real vs fake)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

print("Model loaded!")

Using: cuda


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 128MB/s]


Model loaded!


Training


In [22]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=.001)

NUM_EPOCHS = 5

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels, _ in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Loss: {running_loss/len(train_loader):.4f} | Acc: {correct/total:.4f}")

Epoch 1/5 | Loss: 0.3074 | Acc: 0.8673
Epoch 2/5 | Loss: 0.2612 | Acc: 0.8713
Epoch 3/5 | Loss: 0.2514 | Acc: 0.8713
Epoch 4/5 | Loss: 0.2554 | Acc: 0.8713
Epoch 5/5 | Loss: 0.2508 | Acc: 0.8713


In [23]:
import os

save_path = '/content/resnet50_baseline.pth'
torch.save(model.state_dict(), save_path)
print(f"Model saved to {save_path}")

Model saved to /content/resnet50_baseline.pth


Testing

In [24]:
model = models.resnet50(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 2)
model.load_state_dict(torch.load('/content/resnet50_baseline.pth',
                                  map_location=device))
model = model.to(device)
model.eval()
print("Model loaded!")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Model loaded!


In [26]:
from collections import defaultdict
from sklearn.metrics import classification_report, confusion_matrix, f1_score

model.eval()

video_predictions = defaultdict(list)
video_true_labels = {}

with torch.no_grad():
    for images, labels, video_names in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()

        for video_name, pred, label in zip(video_names, preds, labels.numpy()):
            video_predictions[video_name].append(int(pred))
            video_true_labels[video_name] = int(label)

# Apply majority voting
y_true = []
y_pred = []

for video_name, frame_preds in video_predictions.items():
    fake_votes  = sum(frame_preds)
    total_votes = len(frame_preds)
    final_pred  = 1 if fake_votes > total_votes / 2 else 0
    y_true.append(video_true_labels[video_name])
    y_pred.append(final_pred)

print("\n=== BASELINE RESULTS (Video Level) ===\n")

# Check what's in y_true and y_pred
print(f"y_true: {y_true}")
print(f"y_pred: {y_pred}")
print(f"Unique classes in y_true: {set(y_true)}")
print(f"Unique classes in y_pred: {set(y_pred)}")

print(classification_report(
    y_true, y_pred,
    labels=[0, 1],                              # force both classes
    target_names=['Real (0)', 'Fake (1)'],
    zero_division=0                             # handle missing classes gracefully
))
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))
print(f"\nF1 Score (weighted): {f1_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
print(f"F1 Score (macro):    {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")


=== BASELINE RESULTS (Video Level) ===

y_true: [1, 1]
y_pred: [1, 1]
Unique classes in y_true: {1}
Unique classes in y_pred: {1}
              precision    recall  f1-score   support

    Real (0)       0.00      0.00      0.00         0
    Fake (1)       1.00      1.00      1.00         2

    accuracy                           1.00         2
   macro avg       0.50      0.50      0.50         2
weighted avg       1.00      1.00      1.00         2

Confusion Matrix:
[[0 0]
 [0 2]]

F1 Score (weighted): 1.0000
F1 Score (macro):    1.0000


Eval

In [ ]:

print("\n=== BASELINE RESULTS (Video Level) ===\n")
print(classification_report(
    y_true, y_pred,
    target_names=['Real (0)', 'Fake (1)']
))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))
print()
print(f"F1 Score (weighted): {f1_score(y_true, y_pred, average='weighted'):.4f}")
print(f"F1 Score (macro):    {f1_score(y_true, y_pred, average='macro'):.4f}")